# Silver Processing — Region

Creates `ecommerce.silver.silver_region`, a derived reference dimension mapping every (country, state) pair found in `silver_customers` to a geographic region (34 pairs, 7 countries):

* **India** — compass regions per the user's convention (e.g. `TN` → `South`): North (`DL`, `UP`, `RJ`), South (`KA`, `KL`, `TN`, `TS`), West (`GJ`, `MH`), East (`WB`)
* **United States** — standard census regions: Northeast (`MA`, `NJ`, `NY`), Midwest (`IL`), South (`FL`, `TX`), West (`CA`, `WA`)
* **Australia** — East (`NSW`, `VIC`, `QLD`), West (`WA`)
* **Canada** — West (`BC`, `AB`), Central (`ON`), East (`QC`)
* **United Kingdom** — constituent countries as regions: England, Scotland, Wales, Northern Ireland
* **UAE** — Dubai, Abu Dhabi, Northern Emirates
* **Singapore** — city-state, single region

This table is derived (not file-sourced), so it carries `ingested_at` but no `source_file`. Coverage against `silver_customers` is validated before the write.

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StringType, StructField, StructType

target_table = "ecommerce.silver.silver_region"

# Region mapping for every (country, state) pair present in silver_customers
region_rows = [
    # India — compass regions
    ("IN", "India", "DL", "North"),
    ("IN", "India", "UP", "North"),
    ("IN", "India", "RJ", "North"),
    ("IN", "India", "KA", "South"),
    ("IN", "India", "KL", "South"),
    ("IN", "India", "TN", "South"),
    ("IN", "India", "TS", "South"),
    ("IN", "India", "GJ", "West"),
    ("IN", "India", "MH", "West"),
    ("IN", "India", "WB", "East"),
    # United States — census regions
    ("US", "United States", "MA", "Northeast"),
    ("US", "United States", "NJ", "Northeast"),
    ("US", "United States", "NY", "Northeast"),
    ("US", "United States", "IL", "Midwest"),
    ("US", "United States", "FL", "South"),
    ("US", "United States", "TX", "South"),
    ("US", "United States", "CA", "West"),
    ("US", "United States", "WA", "West"),
    # Australia
    ("AU", "Australia", "NSW", "East"),
    ("AU", "Australia", "VIC", "East"),
    ("AU", "Australia", "QLD", "East"),
    ("AU", "Australia", "WA", "West"),
    # Canada
    ("CA", "Canada", "BC", "West"),
    ("CA", "Canada", "AB", "West"),
    ("CA", "Canada", "ON", "Central"),
    ("CA", "Canada", "QC", "East"),
    # United Kingdom — constituent countries
    ("GB", "United Kingdom", "ENG", "England"),
    ("GB", "United Kingdom", "SCT", "Scotland"),
    ("GB", "United Kingdom", "WLS", "Wales"),
    ("GB", "United Kingdom", "NIR", "Northern Ireland"),
    # UAE
    ("AE", "United Arab Emirates", "DU", "Dubai"),
    ("AE", "United Arab Emirates", "AUH", "Abu Dhabi"),
    ("AE", "United Arab Emirates", "SHJ", "Northern Emirates"),
    # Singapore
    ("SG", "Singapore", "SG", "Singapore"),
]

schema = StructType([
    StructField("country_code", StringType()),
    StructField("country", StringType()),
    StructField("state", StringType()),
    StructField("region", StringType()),
])

df_silver_region = (
    spark.createDataFrame(region_rows, schema)
    .withColumn("ingested_at", current_timestamp())
)

# Validate coverage BEFORE writing: every (country, state) pair in silver_customers
# must exist in the mapping (and vice versa — no unused mapping rows)
distinct_pairs = (
    spark.table("ecommerce.silver.silver_customers")
    .select("country_code", "state")
    .distinct()
)
missing_in_mapping = (
    distinct_pairs.join(df_silver_region, on=["country_code", "state"], how="left_anti")
)
unused_in_mapping = (
    df_silver_region.join(distinct_pairs, on=["country_code", "state"], how="left_anti")
)
print(f"silver_customers pairs not covered by mapping: {missing_in_mapping.count()}")
print(f"mapping rows not used by any customer: {unused_in_mapping.count()}")
assert missing_in_mapping.count() == 0, "Mapping is missing region(s) for pairs in silver_customers"

# Idempotent full load into the silver table
(
    df_silver_region.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_region"))